# Hitcoins 

## Libraries

In [3]:
#setting the environment
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [4]:
import re
import os
import smtplib
import pymongo
import warnings
import numpy as np
import pandas as pd
import datetime as dt
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt
from flatten_json import flatten
from google.cloud import bigquery
from bson.objectid import ObjectId
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
warnings.filterwarnings('ignore')

## Date and Time

In [46]:
end = dt.datetime.today() - dt.timedelta(2)
end = end.replace(hour=18, minute=30, second=0, microsecond=0)
start = end - dt.timedelta(1)
print(start,end,end-start)

2019-06-18 18:30:00 2019-06-19 18:30:00 1 day, 0:00:00


## Users

In [47]:
# cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
#                                  os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
#                                  os.environ['dbname']) 
# c_payment_orders = cursor.superstars.payment_orders
# aw = []
# c_users = cursor.superstars.users      
# for documents in c_users.aggregate([{'$app_data.app_kind' : {'_id' : -1}},  
#                                     {'$unwind':"$app_data"},
#                                     {"$match" : {"app_data.app_kind" : '2' }}]):
#     aw.append(documents)
# dic_flattened = [flatten(d) for d in aw]
# tgpl_users = pd.DataFrame(dic_flattened)
# # tgpl_users = tgpl_users[['_id','app_data_app_kind']]
# # tgpl_users = tgpl_users.fillna(1)
# tgpl_users.head()

In [48]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                                 os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                                 os.environ['dbname'])
aw = []
c_users = cursor.superstars.users
for documents in c_users.find({},{'app_data'}): 
    aw.append(documents)
        
dic_flattened = [flatten(d) for d in aw]
users = pd.DataFrame(dic_flattened)
    
users = users[['_id','app_data_0_app_kind']]
users = users.fillna(1)
tgpl_users = users[users['app_data_0_app_kind']=='2']
del users
tgpl_users.head()

,_id,app_data_0_app_kind
108052,5ccb07d27eeecd71368331d6,2
108094,5ccbd2b8c218a0494b73fe87,2
108134,5ccc1ffd2321b225a12515c3,2
108144,5ccc3d5e6644c97a2275883a,2
108150,5ccc4b91a7799779b258379b,2


## Hitcoins Spent

Mongo DB Connections for user collectable logs

In [31]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname']) 
c_ucl = cursor.superstars.user_collectables_logs

Hitcoins

In [32]:
aw = []
for documents in c_ucl.aggregate([{'$sort' : {'_id' : -1}}, 
                                    {"$match" : {"type" : "HARD_CURRENCY"}}, 
                                    {'$unwind':"$data"},
                                    {"$match" : {"data.created_at" : {"$gte" : start,
                                                                     "$lte" : end}}}]):
                                    
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
df_main = pd.DataFrame(dic_flattened)
df_main.head()

,__v,_id,data__id,data_action_item_id,data_action_item_type,data_catalogue_id,data_created_at,data_quantity,data_reason_type,type,user
0,0,5d0bcef3332aa700156b02d3,5d0bcef3332aa700156b02d4,NaN,NaN,1,2019-06-20 18:22:43.886,-1,QUICK_TRAINING,HARD_CURRENCY,5d0bce32c34132001c9abb81
1,0,5d0bcef3332aa700156b02d3,5d0bcefa332aa700156b02e7,NaN,NaN,1,2019-06-20 18:22:50.701,-1,QUICK_TRAINING,HARD_CURRENCY,5d0bce32c34132001c9abb81
2,0,5d0bcef3332aa700156b02d3,5d0bcf9d332aa700156b363b,NaN,NaN,1,2019-06-20 18:25:33.846,5,LEVEL_UP,HARD_CURRENCY,5d0bce32c34132001c9abb81
3,0,5d0bcef3332aa700156b02d3,5d0bcfb6332aa700156b3b70,NaN,NaN,1,2019-06-20 18:25:58.034,-1,QUICK_TRAINING,HARD_CURRENCY,5d0bce32c34132001c9abb81
4,0,5d0bcef3332aa700156b02d3,5d0bcfbc332aa700156b3b92,NaN,NaN,1,2019-06-20 18:26:04.150,-1,QUICK_TRAINING,HARD_CURRENCY,5d0bce32c34132001c9abb81


### Superstars

In [33]:
df = df_main[~df_main['user'].isin(tgpl_users['_id'])]
df.head()

,__v,_id,data__id,data_action_item_id,data_action_item_type,data_catalogue_id,data_created_at,data_quantity,data_reason_type,type,user
0,0,5d0bcef3332aa700156b02d3,5d0bcef3332aa700156b02d4,NaN,NaN,1,2019-06-20 18:22:43.886,-1,QUICK_TRAINING,HARD_CURRENCY,5d0bce32c34132001c9abb81
1,0,5d0bcef3332aa700156b02d3,5d0bcefa332aa700156b02e7,NaN,NaN,1,2019-06-20 18:22:50.701,-1,QUICK_TRAINING,HARD_CURRENCY,5d0bce32c34132001c9abb81
2,0,5d0bcef3332aa700156b02d3,5d0bcf9d332aa700156b363b,NaN,NaN,1,2019-06-20 18:25:33.846,5,LEVEL_UP,HARD_CURRENCY,5d0bce32c34132001c9abb81
3,0,5d0bcef3332aa700156b02d3,5d0bcfb6332aa700156b3b70,NaN,NaN,1,2019-06-20 18:25:58.034,-1,QUICK_TRAINING,HARD_CURRENCY,5d0bce32c34132001c9abb81
4,0,5d0bcef3332aa700156b02d3,5d0bcfbc332aa700156b3b92,NaN,NaN,1,2019-06-20 18:26:04.150,-1,QUICK_TRAINING,HARD_CURRENCY,5d0bce32c34132001c9abb81


#### Data Wrangling

In [34]:
# filters for hitcoins
hitcoins = df[df['data_catalogue_id']== '1']

# filters for sink
hitcoins = hitcoins[ hitcoins['data_quantity'] < 0 ]
hitcoins['data_quantity'] = -hitcoins['data_quantity']

#Maximum Hitcoins spent
_ = hitcoins.groupby(['data_reason_type','user']).agg({'data_quantity':'sum'}).reset_index()
_ = _.groupby(['data_reason_type']).agg({'data_quantity':'max'}).reset_index()
_

,data_reason_type,data_quantity
0,DAILY_KIT_REFRESH,70
1,DAILY_KIT_REWARDS,171
2,END_TRAINING,407
3,ENERGY_PURCHASE,30
4,KIT_KEYS_PURCHASE,13320
5,QUICK_TRAINING,308
6,SPEEDUP_USE,101


In [35]:
hitcoins = hitcoins.groupby(['data_reason_type']).agg({'data_quantity':'sum','user':pd.Series.nunique}).reset_index()
hitcoins['Hitcoins Spent per User'] = (hitcoins['data_quantity'] / hitcoins['user']).round(1)

hitcoins = pd.merge(hitcoins,_,on='data_reason_type',how='inner')
hitcoins.columns = ['Feature','Total HC Spent','Unique Users','HC Spent per Unique User','Max HC Spent']
hitcoins['Feature'] = hitcoins['Feature'].replace({"DAILY_KIT_REFRESH" : "Daily Deals Refresh",
                                                     "DAILY_KIT_REWARDS" : "Daily Deals Purchase",
                                                     "END_TRAINING" : "Finish Training",
                                                     "ENERGY_PURCHASE" : "Energy Purchase",
                                                     "KIT_KEYS_PURCHASE" : "Sponsor Box Open",
                                                     "QUICK_TRAINING" : "Instant Training",
                                                     "SPEEDUP_USE" : "Speedup Purchase"})

custom_dict = {"Instant Training":0,"Finish Training":1,"Sponsor Box Open":2,"Speedup Purchase":3,
               "Daily Deals Purchase":4,"Daily Deals Refresh":5,"Energy Purchase":6}
hitcoins = hitcoins.iloc[hitcoins['Feature'].map(custom_dict).argsort()]
ss_total = hitcoins['Total HC Spent'].sum()
hitcoins

,Feature,Total HC Spent,Unique Users,HC Spent per Unique User,Max HC Spent
5,Instant Training,1086,63,17.2,308
2,Finish Training,776,51,15.2,407
4,Sponsor Box Open,13620,10,1362.0,13320
6,Speedup Purchase,196,16,12.2,101
1,Daily Deals Purchase,763,58,13.2,171
0,Daily Deals Refresh,160,13,12.3,70
3,Energy Purchase,160,10,16.0,30


In [36]:
hitcoins['Total HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Total HC Spent']), axis=1)
hitcoins['Unique Users'] = hitcoins.apply(lambda x: "{:,}".format(x['Unique Users']), axis=1)
hitcoins['HC Spent per Unique User'] = hitcoins.apply(lambda x: "{:,}".format(x['HC Spent per Unique User']), axis=1)
hitcoins['Max HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Max HC Spent']), axis=1)
ss_hitcoins = hitcoins
hitcoins

,Feature,Total HC Spent,Unique Users,HC Spent per Unique User,Max HC Spent
5,Instant Training,"1,086",63,17.2,308
2,Finish Training,776,51,15.2,407
4,Sponsor Box Open,"13,620",10,"1,362.0","13,320"
6,Speedup Purchase,196,16,12.2,101
1,Daily Deals Purchase,763,58,13.2,171
0,Daily Deals Refresh,160,13,12.3,70
3,Energy Purchase,160,10,16.0,30


### TGPL

In [37]:
df = df_main[df_main['user'].isin(tgpl_users['_id'])]
df.head()

,__v,_id,data__id,data_action_item_id,data_action_item_type,data_catalogue_id,data_created_at,data_quantity,data_reason_type,type,user
921,0,5d049286a47320001256462d,5d0b9c4bc34132001c8f171f,NaN,NaN,1,2019-06-20 14:46:35.185,5,LEVEL_UP,HARD_CURRENCY,5d048073afe99c0011b6bd85
938,0,5d03c3c1914531001894b110,5d0b8dc8c34132001c8b5959,NaN,NaN,1,2019-06-20 13:44:40.222,-72,DAILY_KIT_REWARDS,HARD_CURRENCY,5d03c0ccde1a960011a0b279
939,0,5d03c3c1914531001894b110,5d0b8dcac34132001c8b59e0,NaN,NaN,1,2019-06-20 13:44:42.505,-120,DAILY_KIT_REWARDS,HARD_CURRENCY,5d03c0ccde1a960011a0b279


#### Data Wrangling

In [38]:
# filters for hitcoins
hitcoins = df[df['data_catalogue_id']== '1']

# filters for sink
hitcoins = hitcoins[ hitcoins['data_quantity'] < 0 ]
hitcoins['data_quantity'] = -hitcoins['data_quantity']

#Maximum Hitcoins spent
_ = hitcoins.groupby(['data_reason_type','user']).agg({'data_quantity':'sum'}).reset_index()
_ = _.groupby(['data_reason_type']).agg({'data_quantity':'max'}).reset_index()
_

,data_reason_type,data_quantity
0,DAILY_KIT_REWARDS,192


In [39]:
hitcoins = hitcoins.groupby(['data_reason_type']).agg({'data_quantity':'sum','user':pd.Series.nunique}).reset_index()
hitcoins['Hitcoins Spent per User'] = (hitcoins['data_quantity'] / hitcoins['user']).round(1)

hitcoins = pd.merge(hitcoins,_,on='data_reason_type',how='inner')
hitcoins.columns = ['Feature','Total HC Spent','Unique Users','HC Spent per Unique User','Max HC Spent']
hitcoins['Feature'] = hitcoins['Feature'].replace({"DAILY_KIT_REFRESH" : "Daily Deals Refresh",
                                                     "DAILY_KIT_REWARDS" : "Daily Deals Purchase",
                                                     "END_TRAINING" : "Finish Training",
                                                     "ENERGY_PURCHASE" : "Energy Purchase",
                                                     "KIT_KEYS_PURCHASE" : "Sponsor Box Open",
                                                     "QUICK_TRAINING" : "Instant Training",
                                                     "SPEEDUP_USE" : "Speedup Purchase"})

custom_dict = {"Instant Training":0,"Finish Training":1,"Sponsor Box Open":2,"Speedup Purchase":3,
               "Daily Deals Purchase":4,"Daily Deals Refresh":5,"Energy Purchase":6}
hitcoins = hitcoins.iloc[hitcoins['Feature'].map(custom_dict).argsort()]
tgpl_total = hitcoins['Total HC Spent'].sum()
hitcoins

,Feature,Total HC Spent,Unique Users,HC Spent per Unique User,Max HC Spent
0,Daily Deals Purchase,192,1,192.0,192


In [40]:
hitcoins['Total HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Total HC Spent']), axis=1)
hitcoins['Unique Users'] = hitcoins.apply(lambda x: "{:,}".format(x['Unique Users']), axis=1)
hitcoins['HC Spent per Unique User'] = hitcoins.apply(lambda x: "{:,}".format(x['HC Spent per Unique User']), axis=1)
hitcoins['Max HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Max HC Spent']), axis=1)
tgpl_hitcoins = hitcoins
hitcoins

,Feature,Total HC Spent,Unique Users,HC Spent per Unique User,Max HC Spent
0,Daily Deals Purchase,192,1,192.0,192


## Hitcoins Purchased

In [41]:
cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                             os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                             os.environ['dbname']) 
c_payment_orders = cursor.superstars.payment_orders

In [19]:
aw = []
for documents in c_payment_orders.find({"updated_at" : {"$gte" : start, "$lte" : end}}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
purchased = pd.DataFrame(dic_flattened)

try:
        purchased = purchased[purchased['status']==2]
        purchased = purchased[purchased['currency_code']=='INR']
        purchased = purchased[['items_0_id','items_0_id']]
        purchased.columns = ['id','type']

        data = [[1,50,49], [2,250,199], [3,700,499], [4,2500,1599], [5,6500,3999], [6,14000,7900]] 
        rates = pd.DataFrame(data, columns = ['id', 'hitcoins', 'rates']) 

        purchased = pd.merge (purchased,rates,on='id',how='inner')
        purchased = purchased['hitcoins'].astype(int).sum()
        purchased = "{:,}".format(int(purchased))
except:
        purchased = 0
ss_purchased=purchased
purchased

In [43]:
aw = []
for documents in c_payment_orders.find({"updated_at" : {"$gte" : start, "$lte" : end}}):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
purchased = pd.DataFrame(dic_flattened)

purchased = purchased[purchased['status']==2]
purchased = purchased[purchased['currency_code']!='TGPL']
purchased = purchased[['items_0_id','items_0_id']]
purchased.columns = ['id','type']

data = [[1,50,49], [2,250,199], [3,700,499], [4,2500,1599], [5,6500,3999], [6,14000,7900]] 
rates = pd.DataFrame(data, columns = ['id', 'hitcoins', 'rates']) 

purchased = pd.merge (purchased,rates,on='id',how='inner')
purchased = purchased['hitcoins'].astype(int).sum()

purchased

14000

In [21]:
aw = []
for documents in c_payment_orders.find({"updated_at" : {"$gte" : start, "$lte" : end}}  ):
    aw.append(documents)
dic_flattened = [flatten(d) for d in aw]
purchased = pd.DataFrame(dic_flattened)

try:
        purchased = purchased[purchased['status']==2]
        purchased = purchased[purchased['currency_code']=='TGPL']
        purchased = purchased[['items_0_id','items_0_id']]
        purchased.columns = ['id','type']

        data = [[1,50,49], [2,250,199], [3,700,499], [4,2500,1599], [5,6500,3999], [6,14000,7900]] 
        rates = pd.DataFrame(data, columns = ['id', 'hitcoins', 'rates']) 
        rates['hitcoins'] = rates.apply(lambda x: "{:,}".format(x['hitcoins']), axis=1)

        purchased = pd.merge (purchased,rates,on='id',how='inner')
        purchased = purchased['hitcoins'].astype(int).sum()
except:
        purchased = 0
tgpl_purchased=purchased
tgpl_purchased

50

# Mailers

## Rendering Tables

In [22]:
numeric_col_mask = ss_hitcoins.dtypes.apply(lambda d: issubclass(np.dtype(d).type, np.number))

# Dict used to center the table headers
d = dict(selector="col_heading",
    props=[('text-align', 'center'),('font-weight', 'bold'),('font-size', '14px')])

# Style
style1 = ss_hitcoins.style.set_properties(subset=ss_hitcoins.columns[numeric_col_mask], # right-align the numeric columns and set their width
                        **{'width':'10em', 'text-align':'right'})\
        .set_properties(subset=ss_hitcoins.columns[~numeric_col_mask], # left-align the non-numeric columns and set their width
                        **{'width':'10em', 'text-align':'left'})\
        .set_table_styles([d]).render()

In [23]:
numeric_col_mask = tgpl_hitcoins.dtypes.apply(lambda d: issubclass(np.dtype(d).type, np.number))

# Dict used to center the table headers
d = dict(selector="col_heading",
    props=[('text-align', 'center'),('font-weight', 'bold'),('font-size', '14px')])

# Style
style2 = tgpl_hitcoins.style.set_properties(subset=tgpl_hitcoins.columns[numeric_col_mask], # right-align the numeric columns and set their width
                        **{'width':'10em', 'text-align':'right'})\
        .set_properties(subset=tgpl_hitcoins.columns[~numeric_col_mask], # left-align the non-numeric columns and set their width
                        **{'width':'10em', 'text-align':'left'})\
        .set_table_styles([d]).render()

In [24]:
html_str = """<html>
<head>
<style>

    h2 {
        font-family: Helvetica, Arial, sans-serif;
    }
    table, th, td {
        border: 1px solid black;
        border-collapse: collapse;
    }
    th, td {
        padding: 5px;
        font-family: Helvetica, Arial, sans-serif;
        font-size: 100%;
    }
    tbody tr:nth-child(odd) {background: #eee}
    tbody tr:nth-child(even) {background: #fff}
    table tbody tr td:hover {
        background-color: #fcffb2;
    }
    .row_heading, .blank{
        display: none;
    }
    .col_heading{
        background-color: #99A3A4;
    }
    .row1, .row3, .row5{
        background-color: #EAEDED;
    }
    .row0:hover, .row2:hover, .row4:hover, .row6:hover{
        background-color: #fcffb2;
    }
    .col1, .col2, .col3, .col4, .col5{
        text-align: right;
    }
    .col0, .col1, .col2, .col3, .col4, .col5{
        font-weight: normal;
    }
    .col_heading, .level0, .blank{
        font-weight: bold;
    }
</style>
</head>
<body>
"""

In [25]:
total = ss_total + tgpl_total
purchased = ss_purchased + tgpl_purchased
ss_total = "{:,}".format(int(ss_total))
tgpl_total = "{:,}".format(int(tgpl_total))
ss_purchased = "{:,}".format(int(ss_purchased))
tgpl_purchased = "{:,}".format(int(tgpl_purchased))
total = "{:,}".format(int(total))
purchased = "{:,}".format(int(purchased))

html_str += f"""
<img src="https://d8tuj5f40nouo.cloudfront.net/images/web/landing/logo.png" width="200" height="83">
<h2> Hitcoins Usage: {datetime.strftime(end,'%A, %b %d')} </h2>
<h3>Superstars </h3>
<h4>Total Hitcoins Spent: {ss_total}</h4> 
<h4>Hitcoins Purchased - {ss_purchased} </h4>
{style1}
<br></br>

<h3>TGPL </h3>
<h4>Total Hitcoins Spent: {tgpl_total}</h4> 
<h4>Hitcoins Purchased - {tgpl_purchased} </h4> 
{style2}
</body></html>
"""
subject = f"""Hitcoins: {total} ({purchased})"""

NameError: name 'ss_purchased' is not defined

In [ ]:
gmail_user = os.environ['mail']
gmail_password = os.environ['mail_token']
to = ['isha@hitwicket.com']
sent_from = gmail_user

text = "Please use an html reader"
message = MIMEMultipart(
    "alternative", None, [MIMEText(text), MIMEText(html_str,'html')])

message['From'] = "Analytics <" + os.environ['mail'] + ">"
message['To'] = ','.join(to)
message['Subject'] = subject

s = smtplib.SMTP('smtp.gmail.com', 587) 
s.starttls() 
s.login(gmail_user, gmail_password) 
s.sendmail(sent_from, to, message.as_string())  
s.quit() 

# Final Code 

In [5]:
#!/home/aurora/miniconda3/bin/python

from pathlib import Path
from dotenv import load_dotenv
import re
import os
import smtplib
import pymongo
import warnings
import numpy as np
import pandas as pd
import datetime as dt
import seaborn as sns
from datetime import datetime
import matplotlib.pyplot as plt
from flatten_json import flatten
from google.cloud import bigquery
from bson.objectid import ObjectId
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
env_path = Path('/home/aurora') / '.env'
load_dotenv(dotenv_path=env_path)

try:
    end = dt.datetime.today()
    end = end.replace(hour=18, minute=30, second=0, microsecond=0)
    start = end - dt.timedelta(1)
    print(end)
    
    print('Querying Users Table')
    cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                                 os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                                 os.environ['dbname'])
    aw = []
    c_users = cursor.superstars.users
    for documents in c_users.find({},{'app_data'}): 
        aw.append(documents)

    dic_flattened = [flatten(d) for d in aw]
    users = pd.DataFrame(dic_flattened)

    users = users[['_id','app_data_0_app_kind']]

    tgpl_users = users[users['app_data_0_app_kind']=='2']
    del users
    
    print('Querying Collectables Logs Table')
    cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                                 os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                                 os.environ['dbname']) 
    c_ucl = cursor.superstars.user_collectables_logs
    
    aw = []
    for documents in c_ucl.aggregate([{'$sort' : {'_id' : -1}}, 
                                        {"$match" : {"type" : "HARD_CURRENCY"}}, 
                                        {'$unwind':"$data"},
                                        {"$match" : {"data.created_at" : {"$gte" : start,
                                                                         "$lte" : end}}}]):
                                        
        aw.append(documents)
    dic_flattened = [flatten(d) for d in aw]
    df_main = pd.DataFrame(dic_flattened)
    
    df = df_main[~df_main['user'].isin(tgpl_users['_id'])]
    
    print('Data Wrangling')
    hitcoins = df[df['data_catalogue_id']== '1']
    
    hitcoins = hitcoins[ hitcoins['data_quantity'] < 0 ]
    hitcoins['data_quantity'] = -hitcoins['data_quantity']
    
    _ = hitcoins.groupby(['data_reason_type','user']).agg({'data_quantity':'sum'}).reset_index()
    _ = _.groupby(['data_reason_type']).agg({'data_quantity':'max'}).reset_index()
    
    hitcoins = hitcoins.groupby(['data_reason_type']).agg({'data_quantity':'sum','user':pd.Series.nunique}).reset_index()
    hitcoins['Hitcoins Spent per User'] = (hitcoins['data_quantity'] / hitcoins['user']).round(1)
    
    hitcoins = pd.merge(hitcoins,_,on='data_reason_type',how='inner')
    hitcoins.columns = ['Feature','Total HC Spent','Unique Users','HC Spent per Unique User','Max HC Spent']
    hitcoins['Feature'] = hitcoins['Feature'].replace({"DAILY_KIT_REFRESH" : "Daily Deals Refresh",
                                                         "DAILY_KIT_REWARDS" : "Daily Deals Purchase",
                                                         "END_TRAINING" : "Finish Training",
                                                         "ENERGY_PURCHASE" : "Energy Purchase",
                                                         "KIT_KEYS_PURCHASE" : "Sponsor Box Open",
                                                         "QUICK_TRAINING" : "Instant Training",
                                                         "SPEEDUP_USE" : "Speedup Purchase",
                                                         "TC2_PURCHASE":"Unlocking TC-2"})
    
    custom_dict = {"Instant Training":0,"Finish Training":1,"Sponsor Box Open":2,"Speedup Purchase":3,
                   "Daily Deals Purchase":4,"Daily Deals Refresh":5,"Energy Purchase":6}
    hitcoins = hitcoins.iloc[hitcoins['Feature'].map(custom_dict).argsort()]
    ss_total = hitcoins['Total HC Spent'].sum()
    
    hitcoins['Total HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Total HC Spent']), axis=1)
    hitcoins['Unique Users'] = hitcoins.apply(lambda x: "{:,}".format(x['Unique Users']), axis=1)
    hitcoins['HC Spent per Unique User'] = hitcoins.apply(lambda x: "{:,}".format(x['HC Spent per Unique User']), axis=1)
    hitcoins['Max HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Max HC Spent']), axis=1)
    ss_hitcoins = hitcoins
    
    try:
        df = df_main[df_main['user'].isin(tgpl_users['_id'])]

        hitcoins = df[df['data_catalogue_id']== '1']

        hitcoins = hitcoins[ hitcoins['data_quantity'] < 0 ]
        hitcoins['data_quantity'] = -hitcoins['data_quantity']

        _ = hitcoins.groupby(['data_reason_type','user']).agg({'data_quantity':'sum'}).reset_index()
        _ = _.groupby(['data_reason_type']).agg({'data_quantity':'max'}).reset_index()

        hitcoins = hitcoins.groupby(['data_reason_type']).agg({'data_quantity':'sum','user':pd.Series.nunique}).reset_index()
        hitcoins['Hitcoins Spent per User'] = (hitcoins['data_quantity'] / hitcoins['user']).round(1)

        hitcoins = pd.merge(hitcoins,_,on='data_reason_type',how='inner')
        hitcoins.columns = ['Feature','Total HC Spent','Unique Users','HC Spent per Unique User','Max HC Spent']
        hitcoins['Feature'] = hitcoins['Feature'].replace({"DAILY_KIT_REFRESH" : "Daily Deals Refresh",
                                                             "DAILY_KIT_REWARDS" : "Daily Deals Purchase",
                                                             "END_TRAINING" : "Finish Training",
                                                             "ENERGY_PURCHASE" : "Energy Purchase",
                                                             "KIT_KEYS_PURCHASE" : "Sponsor Box Open",
                                                             "QUICK_TRAINING" : "Instant Training",
                                                             "SPEEDUP_USE" : "Speedup Purchase",
                                                             "TC2_PURCHASE":"Unlocking TC-2"})

        custom_dict = {"Instant Training":0,"Finish Training":1,"Sponsor Box Open":2,"Speedup Purchase":3,
                       "Daily Deals Purchase":4,"Daily Deals Refresh":5,"Energy Purchase":6}
        hitcoins = hitcoins.iloc[hitcoins['Feature'].map(custom_dict).argsort()]
        tgpl_total = hitcoins['Total HC Spent'].sum()


        hitcoins['Total HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Total HC Spent']), axis=1)
        hitcoins['Unique Users'] = hitcoins.apply(lambda x: "{:,}".format(x['Unique Users']), axis=1)
        hitcoins['HC Spent per Unique User'] = hitcoins.apply(lambda x: "{:,}".format(x['HC Spent per Unique User']), axis=1)
        hitcoins['Max HC Spent'] = hitcoins.apply(lambda x: "{:,}".format(x['Max HC Spent']), axis=1)
        tgpl_hitcoins = hitcoins
    except:
        data = [["Daily Deals Refresh",0,0,0,0],
                ["Daily Deals Purchase",0,0,0,0],
                ["Finish Training",0,0,0,0],
                ["Energy Purchase",0,0,0,0],
                ["Sponsor Box Open",0,0,0,0],
                ["Instant Training",0,0,0,0],
                ["Speedup Purchase",0,0,0,0],
                ["Unlocking TC-2",0,0,0,0]] 
        tgpl_hitcoins= pd.DataFrame(data, columns = ['Feature','Total HC Spent','Unique Users','HC Spent per Unique User','Max HC Spent']) 
        tgpl_hitcoins
    print('Getting Purchase Details')
    
    cursor = pymongo.MongoClient("mongodb://" + os.environ['user'] + ':' + 
                                 os.environ['pass'] + '@' + os.environ['db1']  + "/?authSource=" + 
                                 os.environ['dbname']) 
    c_payment_orders = cursor.superstars.payment_orders
    
    aw = []
    for documents in c_payment_orders.find({"updated_at" : {"$gte" : start, "$lte" : end},"status" : 2}):
        aw.append(documents)
    dic_flattened = [flatten(d) for d in aw]
    purchased = pd.DataFrame(dic_flattened)

    try:
            purchased = purchased[~purchased['user_name'].isin(["FirzenYogesh","MrYoBear","goyaala"])]
            purchased = purchased[purchased['currency_code']!='TGPL']
            purchased = purchased[['items_0_id','items_0_id']]
            purchased.columns = ['id','type']

            data = [[1,50,49], [2,250,199], [3,700,499], [4,2500,1599], [5,6500,3999], [6,14000,7900]] 
            rates = pd.DataFrame(data, columns = ['id', 'hitcoins', 'rates']) 

            purchased = pd.merge (purchased,rates,on='id',how='inner')
            purchased = purchased['hitcoins'].astype(int).sum()

    except:
            purchased = 0
    ss_purchased=purchased
    
    aw = []
    for documents in c_payment_orders.find({"updated_at" : {"$gte" : start, "$lte" : end}}  ):
        aw.append(documents)
    dic_flattened = [flatten(d) for d in aw]
    purchased = pd.DataFrame(dic_flattened)

    try:
            purchased = purchased[purchased['status']==2]
            purchased = purchased[purchased['currency_code']=='TGPL']
            purchased = purchased[['items_0_id','items_0_id']]
            purchased.columns = ['id','type']

            data = [[1,50,49], [2,250,199], [3,700,499], [4,2500,1599], [5,6500,3999], [6,14000,7900]] 
            rates = pd.DataFrame(data, columns = ['id', 'hitcoins', 'rates']) 

            purchased = pd.merge (purchased,rates,on='id',how='inner')
            purchased = purchased['hitcoins'].astype(int).sum()
    except:
            purchased = 0
    tgpl_purchased=purchased

    numeric_col_mask = ss_hitcoins.dtypes.apply(lambda d: issubclass(np.dtype(d).type, np.number))
    
    # Dict used to center the table headers
    d = dict(selector="col_heading",
        props=[('text-align', 'center'),('font-weight', 'bold'),('font-size', '14px')])
    
    print('Styling')
    style1 = ss_hitcoins.style.set_properties(subset=ss_hitcoins.columns[numeric_col_mask],**{'width':'10em', 'text-align':'right'})\
            .set_properties(subset=ss_hitcoins.columns[~numeric_col_mask],**{'width':'10em', 'text-align':'left'})\
            .set_table_styles([d]).render()
    
    
    numeric_col_mask = tgpl_hitcoins.dtypes.apply(lambda d: issubclass(np.dtype(d).type, np.number))
    
    d = dict(selector="col_heading",
        props=[('text-align', 'center'),('font-weight', 'bold'),('font-size', '14px')])
    
    style2 = tgpl_hitcoins.style.set_properties(subset=tgpl_hitcoins.columns[numeric_col_mask],**{'width':'10em', 'text-align':'right'})\
            .set_properties(subset=tgpl_hitcoins.columns[~numeric_col_mask],**{'width':'10em', 'text-align':'left'})\
            .set_table_styles([d]).render()
    print('Preparing Mail Content')
    html_str = """<html>
    <head>
    <style>
    
        h2 {
            font-family: Helvetica, Arial, sans-serif;
        }
        table, th, td {
            border: 1px solid black;
            border-collapse: collapse;
        }
        th, td {
            padding: 5px;
            font-family: Helvetica, Arial, sans-serif;
            font-size: 100%;
        }
        tbody tr:nth-child(odd) {background: #eee}
        tbody tr:nth-child(even) {background: #fff}
        table tbody tr td:hover {
            background-color: #fcffb2;
        }
        .row_heading, .blank{
            display: none;
        }
        .col_heading{
            background-color: #CCD1D1;
        }
        .row0:hover, .row2:hover, .row4:hover, .row6:hover, .row8:hover{
            background-color: #fcffb2;
        }
        .col1, .col2, .col3, .col4, .col5{
            text-align: right;
        }
        .col0, .col1, .col2, .col3, .col4, .col5{
            font-weight: normal;
        }
        .col_heading, .level0, .blank{
            font-weight: bold;
        }
        .row1, .row3, .row5, .row7{
            background-color: #EAEDED;
        }
    </style>
    </head>
    <body>
    """
    
    ss_total = "{:,}".format(int(ss_total))
    tgpl_total = "{:,}".format(int(tgpl_total))
    ss_purchased = "{:,}".format(int(ss_purchased))
    tgpl_purchased = "{:,}".format(int(tgpl_purchased))

    html_str += f"""
    <img src="https://d8tuj5f40nouo.cloudfront.net/images/web/landing/logo.png" width="200" height="83">
    <h2> Hitcoins Usage: {datetime.strftime(end,'%A, %b %d')} </h2>
    <h3>Superstars </h3>
    <h4>Total Hitcoins Spent: {ss_total}</h4> 
    <h4>Hitcoins Purchased - {ss_purchased} </h4>
    {style1}
    <br></br>

    <h3>TGPL </h3>
    <h4>Total Hitcoins Spent: {tgpl_total}</h4> 
    <h4>Hitcoins Purchased - {tgpl_purchased} </h4> 
    {style2}
    </body></html>
    
    """
    subject = f"""Hitcoins: {ss_total} ({ss_purchased})  -  TGPL: {tgpl_total} ({tgpl_purchased}) """

    print('Sending Mail')
    gmail_user = os.environ['mail']
    gmail_password = os.environ['mail_token']
    to = ['digest_hitcoins@hitwicket.com','isha@hitwicket.com']
    #to = ['isha@hitwicket.com']
    sent_from = gmail_user
    
    text = "Please use an html reader"
    message = MIMEMultipart(
        "alternative", None, [MIMEText(text), MIMEText(html_str,'html')])
    
    message['From'] = "Analytics <" + os.environ['mail'] + ">"
    message['To'] = ','.join(to)
    message['Subject'] = subject
    
    s = smtplib.SMTP('smtp.gmail.com', 587) 
    s.starttls() 
    s.login(gmail_user, gmail_password) 
    s.sendmail(sent_from, to, message.as_string())  
    s.quit()
    print('Mail Sent')

except:
   print (a)

2019-06-27 18:30:00
Querying Users Table
Querying Collectables Logs Table
Data Wrangling
Getting Purchase Details
Styling
Preparing Mail Content
Sending Mail
Mail Sent


In [ ]:
from slacker import Slacker
   slack = Slacker(os.environ['accio'])
   if slack.api.test().successful:
       print( f"Connected to {slack.team.info().body['team']['name']}.")
   else:
       print('Try Again!')
   slack.chat.post_message(channel='cron',
                           text="Cron-job for Hitcoins Mailer on " + str(dt.date.today()) + " has failed.", 
                           username='accio')